This script applies our final solution for the embedding clustering. The clusters are stored in the database

In [1]:
import pandas as pd
import numpy as np
import ast
import umap
from sklearn.cluster import HDBSCAN

import pandas as pd
from scraper.gpt_prompts import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from db.db_read import convert_dblist_to_df
from db.db_read import setup_database_connection
from db.db_write import User
from db.db_write import EmbeddingCluster
from secret import *


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Daniel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
data = pd.read_csv('mayer_embeddings.csv')

# embedding str to numpy array
def parse_embedding(embedding_str):
    if pd.isna(embedding_str):
        return None
    try:
        return np.array(ast.literal_eval(embedding_str))
    except:
        return None

# only work with main dishes
main_dishes = ['Angebot des Tages', 'Tagesmenü', 'Tagesmenü vegan', 'Tagesmenü vegetarisch','Angebot d. Tages veget.','mensaVital vegan',
                            'Auswahlgericht', 'Auswahlgericht vegan 2', 'Auswahlgericht 2',
                            'Auswahlgericht veget.', 'Auswahlgericht vegan', 'mensaVital vegetarisch']
data = data[data.category.isin(main_dishes)]


# get embedding as array
data['embedding_array'] = data['gpt_embedding'].apply(parse_embedding)

# drop unnecessary columns
data.drop(["date","day","student_price","allergens","additives","guest_price","employee_price","price","ingredients_de","ingredients_en","tokens_used","gpt_embedding"], axis=1,inplace=True)

# clean dish text
def preprocess_text(text):

    text = re.sub(r"\[.*?\]", "", str(text)).strip()
    text = text.lower()
    text = re.sub(r"[^a-zäöüß\s]", "", text)
    text = text.replace("beilage", "").replace("wahl","").replace("mischsalat","").replace("blattsalat","").replace("bunter","")
    text = " ".join(text.split())

    # remove stopwords
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    text = " ".join(filtered_words)

    return text.strip()

# clean dish text
data['meal_clean'] = data['meal'].apply(preprocess_text)
data.dropna(inplace=True)
data.embedding_array.shape

(1695,)

In [3]:
np.stack(data["embedding_array"].values).shape

(1695, 1536)

In [4]:
def umap_hdbscan_clustering(data, umap_params=None, hdbscan_params=None, sample_size=15):

    # set hyperparamters for dimensionality reduction and clustering. We found these to work the best for our data
    umap_params = umap_params or {'n_components': 5, 'random_state': 42}
    hdbscan_params = hdbscan_params or {'min_samples': 10, 'min_cluster_size': 10}
    
    # Initialize UMAP and HDBSCAN
    reducer = umap.UMAP(**umap_params)
    clusterer = HDBSCAN(**hdbscan_params)

    # Fit and transform the embedding matrix using UMAP and cluster the reduced embeddings using HDBSCAN
    embeddings = reducer.fit_transform(np.stack(data["embedding_array"].values))
    hdb = clusterer.fit(embeddings)

    # Assign cluster labels back to the DataFrame
    data['cluster'] = hdb.labels_

    # Sample dishes from each cluster
    sampled_dishes = data.groupby('cluster').apply(lambda x: x.sample(n=sample_size, replace=True)).reset_index(drop=True)[["meal_clean", "cluster"]]
    
    return sampled_dishes,embeddings.shape,hdb

sampled_dishes,emb_shape,hdb = umap_hdbscan_clustering(data)


c:\Users\Daniel\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Daniel\AppData\Local\Temp\ipykernel_9888\1258745411.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_dishes = data.groupby('cluster').apply(lambda x: x.sample(n=sample_size, replace=True)).reset_index(drop=True)[["meal_clean", "cluster"]]


In [5]:
#pd.options.display.max_rows = 4000

#sampled_dishes

In [6]:
cuisine_dict = {
    0: "mexican",
    46: "mexican",
    1: "german",
    3: "german",
    5: "german",
    8: "german",
    10: "german",
    15: "german",
    21: "german",
    27: "german",
    32: "german",
    35: "german",
    36: "german",
    37: "german",
    45: "german",
    50: "german",
    51: "german",
    56: "german",
    57: "german",
    59: "german",
    4: "central european",
    38: "central european",
    48: "central european",
    53: "central european",
    7: "italian",
    11: "italian",
    12: "italian",
    13: "italian",
    20: "italian",
    22: "italian",
    24: "italian",
    40: "italian",
    41: "italian",
    42: "italian",
    43: "italian",
    44: "italian",
    30: "asian",
    31: "asian",
    2: "mediteran",
    44: "mediteran",
    49: "mediteran",
    18: "balkan",
    6: "kreolisch",
    19: "greek",
    23: "thai",
    33: "thai",
    34: "orientalisch",
    52: "french"
}
int_list = [
    0, 46, 1, 3, 5, 8, 10, 15, 21, 27, 32, 35, 36, 37, 45, 50, 51, 56, 57, 59,
    4, 38, 48, 53, 7, 11, 12, 13, 20, 22, 24, 40, 41, 42, 43, 44, 30, 31, 2, 49,
    18, 6, 19, 23, 33, 34, 52
]


relevant_cluster = data[data.cluster.isin(int_list)]
relevant_cluster["cluster_name"] = relevant_cluster.cluster.map(cuisine_dict)
centroids = relevant_cluster.groupby('cluster_name')['embedding_array'].apply(lambda x: np.mean(np.vstack(x), axis=0))

centroid_df = centroids.reset_index(name='centroid')
centroid_df["centroid"] = centroid_df["centroid"].apply(lambda x: x.tolist())
centroid_df["centroid"] = centroid_df["centroid"].apply(lambda x: str(x))
centroid_df

,cluster_name,centroid
0,asian,"[-0.029952381113904074, -0.020184717612497287,..."
1,balkan,"[-0.04680263280476395, -0.015766836420976017, ..."
2,central european,"[-0.03713050101564645, -0.00783074963018643, -..."
3,french,"[-0.0361752076074481, -0.04315755805000663, -0..."
4,german,"[-0.03970345865645622, -0.022047455810511686, ..."
5,greek,"[-0.04718543991966303, -0.019832110378977863, ..."
6,italian,"[-0.032995453836404, -0.027893671327928818, -0..."
7,kreolisch,"[-0.024357426969800144, -0.013418877134304116,..."
8,mediteran,"[-0.06005858337762309, -0.010693222681498703, ..."
9,mexican,"[0.007459329579897383, -0.02267914755329331, -..."


In [7]:
# Write the cluster centers to an SQL table

#engine, Session = setup_database_connection(USER, PASSWORD, HOST, PORT)

#EmbeddingCluster.metadata.create_all(engine)

#table_name = "embedding_clusters"
#centroid_df.to_sql(table_name, engine, if_exists='append',index=False)

In [8]:
regional_cuisine_dict = {
    4: "bavarian",
    10: "bavarian",
    3: "swabian",
    15: "swabian",
    16: "swabian",
    17: "swabian",
    28: "swabian",
    35: "swabian",
    36: "swabian",
    37: "swabian"
}

relevant_cluster = data.copy()
relevant_cluster["cluster_name"] = data.cluster.map(regional_cuisine_dict)
relevant_cluster.replace({"cluster_name": {np.nan: "other"}}, inplace=True)
centroids = relevant_cluster.groupby('cluster_name')['embedding_array'].apply(lambda x: np.mean(np.vstack(x), axis=0))

centroid_df = centroids.reset_index(name='centroid')
centroid_df["centroid"] = centroid_df["centroid"].apply(lambda x: x.tolist())
centroid_df["centroid"] = centroid_df["centroid"].apply(lambda x: str(x))
centroid_df


,cluster_name,centroid
0,bavarian,"[-0.03455281426580862, -0.0026288806275727274,..."
1,other,"[-0.03256773493705877, -0.02100175963744879, -..."
2,swabian,"[-0.033093940652139324, -0.025785752388221405,..."


In [9]:
#table_name = "embedding_clusters"
#centroid_df.to_sql(table_name, engine, if_exists='append',index=False)